# Multi-turn Conversation Evaluation

This notebook evaluates the agent's ability to retain context over multiple turns in a diagnostic dialogue, and verifies the Redis-backed ConversationMemory.

In [ ]:
import uuid
from backend.app.agent.memory import ConversationMemory
from backend.app.agent.react import ReActAgent
from backend.app.agent.registry import PermissionScope

session_id = str(uuid.uuid4())
memory = ConversationMemory(max_turns=2) # Artificially low to force summarization quickly
agent = ReActAgent(agent_scope=PermissionScope.ACTION, max_iterations=3)

def chat(query):
    print(f"\nUSER: {query}")
    history = memory.get_history(session_id)
    result = agent.run(query, history=history)
    print(f"AGENT: {result['answer']}")
    memory.save_turn(session_id, query, result['answer'])

chat("Hi, I'm working on machine_001 today.")
chat("Can you check if there are any current alerts for it?")
chat("Okay, what about the camera snapshot? What does it look like right now?")
chat("Can you summarize everything we've talked about so far?")

print("\n=== HISTORY DUMP ===")
for msg in memory.get_history(session_id):
    print(f"{msg.role}: {msg.parts[0].text[:100]}...")